In [1]:
# Load dataset
import pandas as pd
file_path = "UNITENReview.csv"

df = pd.read_csv(file_path,encoding="latin1")

# Display column content without truncation
review_column = df['Review']

# Display column content without truncation
pd.set_option('display.max_colwidth', None)

# Print just the Review column
print(review_column)

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [2]:
pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [5]:
from deep_translator import GoogleTranslator

# Initialize the translator
# source='auto' allows it to detect Malay automatically
translator = GoogleTranslator(source='auto', target='en')

def translate_to_english(text):
    if not isinstance(text, str) or text.strip() == "":
        return text
    try:
        # Translate the text
        return translator.translate(text)
    except Exception as e:
        # If translation fails, return original text so code doesn't crash
        return text

# Apply translation to the 'Review' column
df["translated_review"] = df["Review"].apply(translate_to_english)

# NOW lowercase the translated version
df["lowercased"] = df["translated_review"].apply(lambda x: x.lower() if isinstance(x, str) else x)

print(df[["Review", "translated_review", "lowercased"]])

In [9]:
import re

def fix_encoding_artifacts(text):
    if not isinstance(text, str):
        return text
    
    # Create a mapping of the "ghost" characters to real punctuation
    mapping = {
        'â€™': "'",  # The one you found first
        'â€™': "'",  # The one you just found
        'â€œ': '"',  # Opening smart quote
        'â€': '"',  # Closing smart quote
        'â€“': '-',  # En dash
        'â€”': '--'   # Em dash
    }
    
    # Replace each artifact
    for artifact, correct_char in mapping.items():
        text = text.replace(artifact, correct_char)
    
    # Remove the # symbol specifically as requested
    text = text.replace('#', '')
    
    # Remove any remaining weird non-ascii symbols but keep normal punctuation
    # This is a "safety net" regex
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    
    return text.strip()

# Apply to your latest column
df["symbols_removed"] = df["lowercased"].apply(fix_encoding_artifacts)

print(df[["lowercased", "symbols_removed"]])

In [12]:
# Replace Contractions
contractions_dict = {
     "wasn't": "was not",
     "isn't": "is not",
     "aren't": "are not",
     "weren't": "were not",
     "doesn't": "does not",
     "don't": "do not",
     "didn't": "did not",
     "can't": "cannot",
     "couldn't": "could not",
     "shouldn't": "should not",
     "wouldn't": "would not",
     "won't": "will not",
     "haven't": "have not",
     "hasn't": "has not",
     "hadn't": "had not",
     "i'm": "i am",
     "you're": "you are",
     "he's": "he is",
     "she's": "she is",
     "it's": "it is",
     "we're": "we are",
     "they're": "they are",
     "i've": "i have",
     "you've": "you have",
     "we've": "we have",
     "they've": "they have",
     "i'd": "i would",
     "you'd": "you would",
     "he'd": "he would",
     "she'd": "she would",
     "we'd": "we would",
     "they'd": "they would",
     "i'll": "i will",
     "you'll": "you will",
     "he'll": "he will",
     "she'll": "she will",
     "we'll": "we will",
     "they'll": "they will",
     "let's": "let us",
     "that's": "that is",
     "who's": "who is",
     "what's": "what is",
     "where's": "where is",
     "when's": "when is",
     "why's": "why is"
}

# Build the regex pattern for contractions
escaped_contractions = [] # List to store escaped contractions

for contraction in contractions_dict.keys():
 escaped_contraction = re.escape(contraction) # Escape special characters (e.g.,apostrophes)
 escaped_contractions.append(escaped_contraction) # Add to list

# Join the escaped contractions with '|'
joined_contractions = "|".join(escaped_contractions)
# Create a regex pattern with word boundaries (\b)
contractions_pattern = r'\b(' + joined_contractions + r')\b'

# Compile the regex
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

# Define a function to replace contractions
def replace_contractions(text):
 # Function to handle each match found
 def replace_match(match):
  matched_word = match.group(0) # Extract matched contraction
  lower_matched_word = matched_word.lower() # Convert to lowercase
  expanded_form = contractions_dict[lower_matched_word] # Get full form from dictionary
  return expanded_form # Return the expanded form

  expanded_form = contractions_dict[lower_matched_word] # Get full form fromdictionary
  return expanded_form # Return the expanded form

# 1. Define the function
def replace_contractions(text):
    if not isinstance(text, str): return text
    def replace_match(match):
        return contractions_dict.get(match.group(0).lower(), match.group(0))
    return compiled_pattern.sub(replace_match, text)

# 2. CREATE the column (NOT indented)
df["contractions_replaced"] = df["symbols_removed"].apply(replace_contractions)

# 3. NOW you can print it
print(df["contractions_replaced"])


0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
# Remove punctuations and special characters
import string
# Function to remove punctuation
def remove_punctuation(text):
 return text.translate(str.maketrans('', '', string.punctuation))
# Apply the function to the column
df["punctuations_removed"] = df["contractions_replaced"].apply(remove_punctuation)
# Display column content without truncation
pd.set_option('display.max_colwidth', None) # Set to None for unlimited width
print(df["punctuations_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [14]:
# Remove numbers
def remove_numbers(text):
 return re.sub(r'\d+', '', text) # Removes all numeric characters
# Apply the function to the column
df["numbers_removed"] = df["punctuations_removed"].apply(remove_numbers)
# Display column content without truncation
pd.set_option('display.max_colwidth', None) # Set to None for unlimited width
print(df["numbers_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [15]:
# Correct spelling mistakes
from autocorrect import Speller
# Initialize spell checker
spell = Speller(lang='en')
# Function to correct spelling
def correct_spelling(text):
 return spell(text) # Apply correction
# Apply the function to the column
df["spelling_corrected"] = df["numbers_removed"].apply(correct_spelling)
# Display column content without truncation
pd.set_option('display.max_colwidth', None) # Set to None for unlimited width
print(df["spelling_corrected"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [16]:
# Remove stopwords
import nltk
from nltk.corpus import stopwords
# Download stopwords if not already downloaded
nltk.download('stopwords')
# Define stopwords list
stop_words = set(stopwords.words('english'))
# Function to remove stopwords
def remove_stopwords(text):
 words = text.split() # Split text into words
 filtered_words = [] # Create an empty list to store words after stopword removal
 for word in words: # Loop through each word in the list of words
  lower_word = word.lower() # Convert the word to lowercase for uniform comparison
  if lower_word not in stop_words: # Check if the lowercase word is NOT in the stopwords list 
      if lower_word not in stop_words: # Check if the lowercase word is NOT in thestopwords list
        filtered_words.append(word) # If it's not a stopword, add it to the filtered list

      return " ".join(filtered_words) # Join words back into a sentence


# Apply the function to the column
df["stopwords_removed"] = df["spelling_corrected"].apply(remove_stopwords)

# Display column content without truncation
pd.set_option('display.max_colwidth', None) # Set to None for unlimited width
print(df["stopwords_removed"])

0               im
1               im
2          neutral
3            would
4           united
5          neutral
6           regret
7          opinion
8           united
9            great
10          united
11            good
12          united
13        honestly
14          united
15             bad
16           faced
17          united
18          united
19           first
20         opinion
21          united
22             bad
23          united
24      management
25          insist
26            good
27          honest
28          united
29        honestly
30       residency
31           great
32         opinion
33            nach
34            good
35          united
36              go
37            well
38            good
39           final
40           great
41           first
42    academically
43          energy
44          united
45            name
46            feel
47        moderate
48      facilities
49          united
50             old
51            feel
Name: stopwo

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
